# Breast Cancer Risk Prediction Pipeline

**Authors:** Suhaan Thayyil & Eshaan Nidee  
**Project:** ISEF 2026

This notebook implements the full end-to-end pipeline:
1. Data Loading (TCGA & GSE96058)
2. Feature Engineering (Pathway Scores)
3. Model Training (Elastic Net, Random Forest, Gradient Boosting)
4. Cross-Validation & External Validation
5. Explainability (SHAP)

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as seaborn
import shap

# Ensure we are in the project root to import src
import sys
sys.path.append('..')

# Import project modules
from src.data_loader import load_data
from src.preprocessing import clean_and_normalize
from src.features import calculate_pathway_scores, add_ratio_features
from src.models import train_xgboost

print("Libraries loaded successfully.")

## 1. Load Preprocessed Data

In [ ]:
# Load data from the data/ directory
data_dir = "../data"

print("Loading TCGA data...")
tcga_features = pd.read_csv(os.path.join(data_dir, "02_tcga_feature_matrix.csv"))
print(f"TCGA Shape: {tcga_features.shape}")

print("Loading Clinical Data...")
tcga_clin = pd.read_csv(os.path.join(data_dir, "01_tcga_clinical.csv"))

## 2. Model Training

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score, accuracy_score

X = tcga_features.drop(['sample_id', 'high_risk', 'time_to_event', 'event_status'], axis=1, errors='ignore')
y = tcga_features['high_risk']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

model = GradientBoostingClassifier(n_estimators=300, learning_rate=0.05, max_depth=3, random_state=42)
model.fit(X_train, y_train)

probs = model.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, probs)
print(f"TCGA Test AUC: {auc:.4f}")

## 3. Explainability (SHAP)

In [ ]:
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

shap.summary_plot(shap_values, X_test)